# LangChain: A Revision Notebook
### Models, structured output, tools, agents, streaming, middleware, HITL, and summarization

**Goal:** understand the progression from a direct model call to a tool-using agent, then add human approval and context management.

**Models used in this notebook:**

- Ollama: `qwen2.5-coder:3b`
- Gemini: `gemini-2.5-flash-lite`

> **Before you run:** install the packages in section 0, set `GOOGLE_API_KEY` in your environment, and make sure Ollama is running if you want to use the local model.


## 0. Install

```bash
pip install -U langchain langchain-ollama langchain-google-genai
```

For HITL and checkpointing:

```bash
pip install -U langgraph
```

Ollama:

```bash
ollama pull qwen2.5-coder:3b
```

Gemini: set `GOOGLE_API_KEY` in your environment.


In [ ]:
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.agents import create_agent
from langchain.tools import tool

from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    SummarizationMiddleware,
)

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


## 1. Models

Create the two model objects used throughout the notebook. Gemini is used for most examples; section 11 repeats the agent with Ollama.


In [ ]:
ollama_model = ChatOllama(
    model="qwen2.5-coder:3b",
    temperature=0,
)

gemini_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
)


## 2. `model.invoke()`

This is the basic LLM call.

| Aspect | Behavior |
|---|---|
| Input | Text or messages |
| Output | `AIMessage` |
| Agent | No |
| Tools | No |
| Agent loop | No |


In [ ]:
response = gemini_model.invoke(
    "Explain gradient descent in one sentence."
)

print(response.content)


# Same thing with Ollama:

response = ollama_model.invoke(
    "Explain gradient descent in one sentence."
)

print(response.content)


## 3. Structured output

Define a Pydantic schema, then convert the model into a structured-output model.

`model.with_structured_output(...)` means: "I want the model's response to follow this schema."

This is **not** an agent.


In [ ]:
class Weather(BaseModel):
    city: str = Field(description="City name")
    temperature: float = Field(description="Temperature in Celsius")
    condition: str = Field(description="Weather condition")


In [ ]:
structured_gemini = gemini_model.with_structured_output(Weather)

response = structured_gemini.invoke(
    "The weather in Kolkata is 31 degrees Celsius and sunny."
)

print(response)


The result is a Pydantic object:

```python
Weather(
    city="Kolkata",
    temperature=31,
    condition="sunny"
)
```


## 4. Tool

The `@tool` decorator makes the conversion explicit and returns a LangChain tool object. It is not strictly necessary for simple functions: `create_agent(..., tools=[get_weather])` can convert the function itself.


In [ ]:
@tool
def get_weather(city: str):
    """Get the weather for a city."""
    print(">>> TOOL EXECUTED")
    return f"It is sunny in {city} today."


Without the decorator, the same function can still be passed to an agent:

```python
def get_weather(city: str):
    """Get the weather for a city."""
    return f"It is sunny in {city} today."

create_agent(..., tools=[get_weather])
```


## 5. Model tool calling with `bind_tools()`

`bind_tools()` gives the model the ability to **request** a tool. It does not create the agent loop.

```text
user
  ↓
LLM
  ↓
tool call
END
```

You would have to execute the tool yourself and send the `ToolMessage` back to the model. `create_agent()` does this loop automatically.


In [ ]:
model_with_tools = gemini_model.bind_tools(
    [get_weather]
)

response = model_with_tools.invoke(
    "What's the weather in Kolkata?"
)

print(response)

print(response.tool_calls)


## 6. Agent

`create_agent()` builds the automatic model-tool loop.

```text
USER
  ↓
MODEL
  ↓
"I should call get_weather"
  ↓
TOOL
  ↓
get_weather("Kolkata")
  ↓
TOOL RESULT
  ↓
MODEL
  ↓
FINAL ANSWER
```

This is the major difference between `model.bind_tools()` and `create_agent(..., tools=[...])`.


In [ ]:
agent = create_agent(
    model=gemini_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?",
            }
        ]
    }
)

print(result)


## 7. Agent with Pydantic structured output

For an agent, use `response_format=...`. Do **not** use `agent.with_structured_output(...)`.

The structured result is returned in `result["structured_response"]`.


In [ ]:
class WeatherAnswer(BaseModel):
    city: str
    weather: str
    recommendation: str


In [ ]:
structured_agent = create_agent(
    model=gemini_model,
    tools=[get_weather],
    response_format=WeatherAnswer,
)

result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me the weather in Kolkata and what I should do today.",
            }
        ]
    }
)

print(result["structured_response"])


The result looks like:

```python
WeatherAnswer(
    city="Kolkata",
    weather="Sunny",
    recommendation="..."
)
```

| Object | Structured output API |
|---|---|
| Model | `model.with_structured_output(Weather)` |
| Agent | `create_agent(..., response_format=Weather)` |
| Result | `result["structured_response"]` |


## 8. `agent.invoke()`

`invoke()` sends one input, runs one complete execution, and returns the final state.


In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello!",
            }
        ]
    }
)

print(result)


## 9. `agent.stream()`

`stream()` lets you observe execution as it happens.


In [ ]:
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?",
            }
        ]
    },
    stream_mode="updates",
):
    print(chunk)


Conceptually, `stream_mode="updates"` emits:

```text
MODEL UPDATE
     ↓
TOOL UPDATE
     ↓
MODEL UPDATE
```

Another useful mode is `"messages"`, which is useful for token/message streaming:

```python
for chunk in agent.stream(
    {...},
    stream_mode="messages",
):
    print(chunk)
```


## 10. `agent.batch()`

`batch()` runs many inputs.


In [ ]:
results = agent.batch(
    [
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in Kolkata?",
                }
            ]
        },
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in Delhi?",
                }
            ]
        },
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in Mumbai?",
                }
            ]
        },
    ]
)


for result in results:
    print(result["messages"][-1].content)


| Method | Behavior |
|---|---|
| `invoke()` | One input |
| `batch()` | Many inputs |
| `stream()` | Execution streamed as it happens |


## 11. The same agent with Ollama

The LangChain agent API is the same. The only change is the model object: `gemini_model` versus `ollama_model`.

Tool-calling reliability depends on whether the model and provider properly support tool calling. A Qwen experiment demonstrated this. If a local model returns JSON-like text instead of a structured tool call, it generated text such as:

```text
content = '{"name": "get_weather", ...}'
tool_calls = []
```

In that case, the model produced JSON-like text instead of a proper structured tool call.


In [ ]:
ollama_agent = create_agent(
    model=ollama_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)


result = ollama_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?",
            }
        ]
    }
)


print(result)


## 12. Human-in-the-loop

HITL means **Human In The Loop**. It is used when the model proposes a dangerous or actionable tool call:

```text
MODEL proposes dangerous/actionable tool call
           ↓
      PAUSE EXECUTION
           ↓
      HUMAN REVIEWS
           ↓
  approve / edit / reject
           ↓
      execution continues
```

A checkpointer is required because the agent must remember its state while execution is paused. In this example, `send_email` requires approval; `get_weather` is safe and does not.


In [ ]:
@tool
def send_email(
    to: str,
    subject: str,
    body: str,
):
    """Send an email."""
    print(">>> EMAIL TOOL EXECUTED")
    return f"Email sent to {to}"


In [ ]:
hitl_agent = create_agent(
    model=gemini_model,
    tools=[
        get_weather,
        send_email,
    ],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "get_weather": False,
            }
        )
    ],
    checkpointer=InMemorySaver(),
)


The thread ID identifies the paused/resumable conversation.


In [ ]:
config = {
    "configurable": {
        "thread_id": "demo-thread-1"
    }
}


Start the agent.


In [ ]:
result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to boss@example.com "
                    "saying I will be late today."
                ),
            }
        ]
    },
    config=config,
    version="v2",
)


print(result)


At this point the agent can interrupt before executing `send_email`.

## 13. HITL action: approve

Human says: "Yes, execute exactly what the agent proposed."

If the model proposed `send_email(to="boss@example.com", ...)` and the human approves, the exact same tool call executes.


In [ ]:
result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config,
    version="v2",
)


## 14. HITL action: edit

Imagine the model proposed `to="boss@example.com"` and the human wants `to="manager@example.com"`.

Instead of rejecting everything, modify the action, then execute the modified version.


In [ ]:
result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {
                            "to": "manager@example.com",
                            "subject": "Running late",
                            "body": "I will be late today.",
                        },
                    },
                }
            ]
        }
    ),
    config=config,
    version="v2",
)


## 15. HITL action: reject

Human says: "No. Don't send this email."

The tool does **not** execute. Instead, the rejection and feedback are passed back into the agent conversation. The model can then decide what to do next.


In [ ]:
result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": (
                        "Do not send this email. "
                        "Ask me before sending anything."
                    ),
                }
            ]
        }
    ),
    config=config,
    version="v2",
)


## HITL: the three core actions

| Action | Behavior |
|---|---|
| Approve | Execute exactly what the model proposed |
| Edit | Modify the proposed tool call, then execute the modified version |
| Reject | Do not execute; give feedback/reason to the model |

Current LangChain also supports `"respond"` for special ask-user style tools, where the human supplies the tool result directly instead of executing the tool. For ordinary tool-approval workflows, remember **approve**, **edit**, and **reject**.


## 16. Summarization middleware

The conversation keeps growing:

```text
User → AI → User → Tool → AI → User → ...
```

Eventually the context window becomes too large. `SummarizationMiddleware` automatically summarizes older conversation history when token usage approaches a configured threshold.

In the example, `trigger=("tokens", 4000)` triggers summarization when the conversation reaches approximately 4,000 tokens, and `keep=("messages", 20)` keeps the most recent 20 messages untouched.


In [ ]:
summarized_agent = create_agent(
    model=gemini_model,
    tools=[
        get_weather,
    ],
    middleware=[
        SummarizationMiddleware(
            model=gemini_model,
            trigger=("tokens", 4000),
            keep=("messages", 20),
        )
    ],
)


Use the agent normally.


In [ ]:
result = summarized_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Remember that I prefer concise answers.",
            }
        ]
    }
)


You do not manually call `summarize()`. Middleware watches the conversation automatically.


## 17. What summarization middleware actually does

```text
BEFORE:
Message 1
Message 2
Message 3
...
Message 100
     ↓ token threshold reached
SummarizationMiddleware
     ↓
OLD MESSAGES → SUMMARY
RECENT MESSAGES → kept intact
```

Conceptually:

```text
[OLD 1]
[OLD 2]
[OLD 3]
   ↓
SUMMARY
   +
[RECENT messages]
```

The model still gets the important context without carrying every old message forever. LangChain also takes care to preserve AI/Tool message relationships when summarizing.


## 18. Everything in one mental model

```text
                LANGCHAIN

      ┌──────────────────────────┐
      │        MODEL             │
      │                          │
      │ model.invoke()           │
      │ model.bind_tools()       │
      │ model.with_structured_   │
      │     output(Pydantic)     │
      └────────────┬─────────────┘
                   │
                   ▼
      ┌──────────────────────────┐
      │        AGENT             │
      │                          │
      │ create_agent()           │
      │ invoke()                 │
      │ stream()                 │
      │ batch()                  │
      │ model ↔ tools loop       │
      └────────────┬─────────────┘
                   │
         ┌─────────┴─────────┐
         ▼                   ▼
      TOOLS                MIDDLEWARE
                             │
                   ┌─────────┴──────────┐
                   ▼                    ▼
                 HITL              SUMMARIZATION
                   │                    │
            approve/edit/reject     compress old
                                   conversation
```


## 19. Revision cheat sheet

| Area | API | What it does |
|---|---|---|
| Model | `model.invoke(...)` | Direct LLM call |
| Model + structured output | `model.with_structured_output(MyPydanticClass)` | Model returns structured Pydantic data |
| Model + tools | `model.bind_tools([tools])` | Model can request tools; you handle execution and the loop |
| Agent | `create_agent(model=model, tools=[tools])` | Model plus automatic tool execution loop |
| Agent + structured output | `create_agent(..., response_format=MyPydanticClass)` | Agent runs tools and returns a final structured response in `result["structured_response"]` |
| Agent invoke | `agent.invoke(input)` | One complete run |
| Agent stream | `agent.stream(input)` | Stream execution/results |
| Agent batch | `agent.batch([input1, input2, input3])` | Execute multiple inputs |
| HITL | `HumanInTheLoopMiddleware(...)` | Pause before selected tools |
| Approve | HITL decision | Execute the original action |
| Edit | HITL decision | Execute the modified action |
| Reject | HITL decision | Do not execute; give feedback to the model |
| Summarization | `SummarizationMiddleware(...)` | Automatically compress old conversation, preserve recent messages, control context growth |


## 20. The one-line memory trick

| Concept | One-line reminder |
|---|---|
| Model | "Give me an answer." |
| `bind_tools` | "You may ask for a tool." |
| Agent | "I'll run the tools and keep looping." |
| Structured output | "Give me the answer in this schema." |
| Stream | "Show me the execution as it happens." |
| Batch | "Do this for many inputs." |
| HITL | "Stop before dangerous tools and ask a human." |
| Summarization | "The conversation is getting too big; compress old history while keeping recent context." |


## Self-test

1. What is the difference between `model.invoke()` and `agent.invoke()`?
2. When would you use `model.with_structured_output(...)` instead of `response_format=...`?
3. What does `bind_tools()` do, and what does it not do?
4. Where does an agent's structured response appear?
5. What is the difference between `invoke()`, `stream()`, and `batch()`?
6. Why does HITL need a checkpointer?
7. What do approve, edit, and reject do?
8. What problem does `SummarizationMiddleware` solve?


## Official references

- [LangChain documentation](https://docs.langchain.com)
- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
